# Codification 1 : la codification 1 comme classe à prédire

**Variante de [`01e_optimisation_tout_regroupe.ipynb`](01e_optimisation_tout_regroupe.ipynb)** (montants et plafond au logarithme signé, codifications regroupées en -2, -1, 0, 2 et plus, même boucle d'optimisation, départ aux meilleurs réglages du 01e). Journal : [`tableau_resultats.md`](tableau_resultats.md).

**Question** : la banque pose-t-elle les codifications 1 selon une logique que le modèle peut apprendre ? Ici, le 1 n'est plus mis de côté : c'est **une cinquième classe** à prédire, à côté de -2, -1, 0 et 2 et plus. Si le modèle retrouve les 1 avec une bonne précision, à partir des mois précédents et des montants, le 1 est une suite logique de certaines situations : la règle de la banque est en partie dans les données. S'il n'y arrive pas, le 1 dépend d'autre chose que ce que le dataset montre. `dpnm` n'est utilisé à aucun moment.

**Données** : niveau 0 (`data/creditcard_pret_ingestion.csv`), sans `dpnm`, sans `ID`, sans données personnelles ; **les 28 clients qui ont une codification 1 d'avril à août sont retirés** : il ne reste que des codifications 1 de septembre.

**Découpage propre à cet essai** : 80/20 sur tous les clients restants, `stratify` sur `PAY_1`, `random_state=42`. Il diffère de celui des notebooks 01 à 01f, puisque les clients codifiés 1 y entrent ; le test de l'hypothèse « code d'attente » garde le sien. Le test de cet essai n'est pas lu ici.

**Score optimisé** : la moyenne des taux de bonne famille des **cinq** familles (-2, -1, 0, 1, 2 et plus), chaque famille pesant autant. Même boucle, même limite de surapprentissage (7 % relatif), même alerte de variance (2 points).

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Clients avec une codification 1 d'avril à août : retirés, il ne reste que des codifications 1 de septembre
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
df = df[~anciens_1]
print(f"Clients retirés (codification 1 d'avril à août) : {int(anciens_1.sum())} ; clients restants : {len(df)} ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

Clients du niveau 0 : 30000 ; colonnes : 24 (dpnm retirée) ; PAY_1 = 1 : 3688
Clients retirés (codification 1 d'avril à août) : 28 ; clients restants : 29972 ; PAY_1 = 1 : 3660


## 2. Le découpage propre à cet essai

**Cible** : `PAY_1`, codifications 1 comprises. **Variables** : les mêmes que dans le 01e (codifications d'avril à août, soldes, paiements, plafond). Tous les clients restants, coupés en 80/20 (train / test, `stratify` sur `PAY_1`, `random_state=42`).

In [2]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
assert int((df[PAY[1:]] == 1).sum().sum()) == 0

X_train, X_test, y_train, y_test = train_test_split(df[VARIABLES], df["PAY_1"], test_size=0.2, stratify=df["PAY_1"], random_state=42)
CLASSES = sorted(df["PAY_1"].unique())
repartition = pd.DataFrame({"Train": y_train.value_counts(), "Test": y_test.value_counts()}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"Clients : train {len(X_train)}, test {len(X_test)} (de côté)")
display(repartition)

Clients : train 23977, test 5995 (de côté)


,Train,Test
PAY_1,,
-2,2207,552
-1,4549,1137
0,11789,2948
1,2928,732
2,2133,534
3,258,64
4,61,15
5,21,5
6,9,2


## 3. Regroupement des retards

Comme dans le 01e : la cible `PAY_1` et les variables `PAY_2` à `PAY_6` sont regroupées, les codifications de 2 à 8 deviennent toutes **2** (retard). La codification 1 reste une classe à part : il reste cinq classes, -2, -1, 0, 1, 2. Le regroupement se fait en mémoire ; le fichier d'origine n'est pas modifié.

In [3]:
def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 et 1 inchangés
    return codifications.where(codifications < 2, 2)

y_train, y_test = regrouper(y_train), regrouper(y_test)
for col in PAY[1:]:
    X_train[col], X_test[col] = regrouper(X_train[col]), regrouper(X_test[col])
CLASSES = sorted(y_train.unique())
assert CLASSES == [-2, -1, 0, 1, 2], CLASSES
print(f"Cible PAY_1 regroupée : classes {CLASSES}")
print(y_train.value_counts().reindex(CLASSES).rename("Train").to_frame().join(y_test.value_counts().rename("Test")))
print("Variables PAY_2 à PAY_6 regroupées : valeurs", sorted(pd.unique(X_train[PAY[1:]].values.ravel())))

Cible PAY_1 regroupée : classes [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]
       Train  Test
PAY_1             
-2      2207   552
-1      4549  1137
 0     11789  2948
 1      2928   732
 2      2504   626
Variables PAY_2 à PAY_6 regroupées : valeurs [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2)]


## 4. Le score optimisé : la moyenne des cinq familles

La moyenne des taux de bonne famille de -2, -1, 0, **1** et 2 et plus : chaque famille pèse autant, la codification 1 comprise.

In [4]:
# ==============================================================================
# Score optimisé : "familles équilibrées" (moyenne des cinq familles, codification 1 comprise)
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from sklearn.base import clone
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [-2, -1, 0, 1, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 1: "1", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 et 1 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
scoreur = make_scorer(score_familles)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des montants dus, payés et du plafond, comme dans 01b
MONTANTS_DUS_PAYES = [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"]

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur ; nombres (déjà au logarithme) : mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

Score optimisé : familles équilibrées ; surapprentissage max 7.0 % relatif ; alerte si écart-type des plis > 2.0 points


## 5. Les réglages explorés

Pour chaque modèle : les **échelles** (paramètres ordonnés, explorés par la boucle autour d'une valeur de départ : ici, **le meilleur réglage du 01e**) et les **choix** (paramètres sans ordre, tous essayés à chaque tour). Les autres paramètres gardent leur valeur par défaut, sauf ceux indiqués en commentaire.

- **Régression logistique** : force de la régularisation `C` (petit = forte régularisation) ; poids des classes (`class_weight` : aucun, ou équilibré pour compenser les codifications rares) ; **type de pénalité** (`l1_ratio` : 0 = L2, qui réduit tous les coefficients ; 1 = L1, qui peut **annuler** les variables inutiles, par exemple des mois trop anciens ; 0,5 = mélange des deux), avec le solveur `saga`, le seul qui accepte la pénalité L1 pour plusieurs classes.
- **Arbre de décision** : profondeur maximale (plafonnée à 30), taille minimale des feuilles (grande = règles plus générales) ; poids des classes ; critère de coupe (`gini` ou `entropy`).
- **RandomForest** : nombre d'arbres, profondeur maximale (plafonnée à 30, sans profondeur illimitée), taille minimale des feuilles ; part des variables essayées à chaque coupe (`max_features`) ; poids des classes.
- **MLP** : taille des couches cachées, régularisation `alpha` ; arrêt anticipé (`early_stopping`) pour éviter d'apprendre par cœur.
- **CatBoost** : réglé comme dans le machine learning de la deuxième itération (même pipeline que les arbres, codifications en une colonne par valeur ; tous les cœurs ; 1 000 itérations). La boucle explore la profondeur, le `learning_rate` et `l2_leaf_reg` ; poids des classes : `Balanced` ou `SqrtBalanced` (plus doux), toujours actifs, CatBoost refusant la valeur « aucun » passée explicitement.

In [5]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(solver="saga", max_iter=5000)),   # saga : seul solveur qui accepte la pénalité L1 en multiclasse
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 10},   # meilleur réglage du 01e
        "choix": {"logisticregression__class_weight": [None, "balanced"],
                  "logisticregression__l1_ratio": [0, 0.5, 1]},   # 0 : pénalité L2 (réduit les coefficients) ; 1 : L1 (peut en annuler) ; 0,5 : mélange
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, 30],   # plafond à 30 (pas de profondeur illimitée)
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 8, "decisiontreeclassifier__min_samples_leaf": 2},   # meilleur réglage du 01e
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, 30],   # plafond à 30 (pas de profondeur illimitée)
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 500, "randomforestclassifier__max_depth": 25, "randomforestclassifier__min_samples_leaf": 5},   # meilleur réglage du 01e
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (128, 64), "mlpclassifier__alpha": 1e-4},   # meilleur réglage du 01e
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 5, "catboostclassifier__learning_rate": 0.03, "catboostclassifier__l2_leaf_reg": 5},   # meilleur réglage du 01e
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

Modèles : ['Régression logistique', 'Arbre de décision', 'RandomForest', 'MLP', 'CatBoost']


## 6. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch en validation croisée (5 plis) sur la grille courante, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Une ligne à la fin de chaque tour (réglages essayés, réglages admis, meilleur admis, durée du tour), pour suivre l'avancement ; puis, pour chaque modèle, une ligne : son meilleur réglage admis, son score en validation croisée et la raison de l'arrêt de la boucle.

In [6]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        debut_tour = time.perf_counter()
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=scoreur, cv=cv, n_jobs=espace["n_jobs"], return_train_score=True)
        recherche.fit(X_train, y_train, **espace.get("fit_params", {}))
        r = recherche.cv_results_
        res = pd.DataFrame({
            "tour": tour,
            "params": r["params"],
            "validation": r["mean_test_score"] * 100,
            "écart-type des plis": r["std_test_score"] * 100,
            "train": r["mean_train_score"] * 100,
        })
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            print(f"{nom}, tour {tour} : {len(res)} réglages, aucun admis ({round(time.perf_counter() - debut_tour)} s)", flush=True)
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        print(f"{nom}, tour {tour} : {len(res)} réglages, {len(admis)} admis ; meilleur admis {admis['validation'].max():.2f} % "
              f"({libelle(meilleur)}) ({round(time.perf_counter() - debut_tour)} s)", flush=True)
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    historique = pd.concat(lignes, ignore_index=True)
    return historique, raison, round(time.perf_counter() - debut)

resultats = {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        print(f"{nom} : aucun réglage admis ; arrêt : {raison} ({duree} s)")
    else:
        m = admis.loc[admis["validation"].idxmax()]
        print(f"{nom} : {m['validation']:.2f} % ± {m['écart-type des plis']:.2f} ({libelle(m['params'])}) ; arrêt : {raison} ({duree} s)")

Régression logistique, tour 1 : 18 réglages, 18 admis ; meilleur admis 75.66 % (C=10, class_weight=balanced, l1_ratio=1) (75 s)


Régression logistique : 75.66 % ± 0.48 (C=10, class_weight=balanced, l1_ratio=1) ; arrêt : tour 1 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (75 s)


Arbre de décision, tour 1 : 36 réglages, 35 admis ; meilleur admis 76.00 % (class_weight=balanced, criterion=entropy, max_depth=8, min_samples_leaf=5) (2 s)


Arbre de décision, tour 2 : 36 réglages, 36 admis ; meilleur admis 76.30 % (class_weight=balanced, criterion=gini, max_depth=8, min_samples_leaf=10) (2 s)


Arbre de décision, tour 3 : 36 réglages, 36 admis ; meilleur admis 76.30 % (class_weight=balanced, criterion=gini, max_depth=8, min_samples_leaf=10) (2 s)


...

MLP, tour 4 : 9 réglages, 9 admis ; meilleur admis 76.89 % (alpha=0.1, hidden_layer_sizes=(128, 64)) (15 s)


MLP : 76.89 % ± 0.29 (alpha=0.1, hidden_layer_sizes=(128, 64)) ; arrêt : tour 4 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (57 s)


CatBoost, tour 1 : 54 réglages, 51 admis ; meilleur admis 77.71 % (auto_class_weights=Balanced, depth=4, l2_leaf_reg=3, learning_rate=0.05) (357 s)


CatBoost, tour 2 : 54 réglages, 44 admis ; meilleur admis 77.71 % (auto_class_weights=Balanced, depth=4, l2_leaf_reg=3, learning_rate=0.05) (256 s)


CatBoost : 77.71 % ± 0.57 (auto_class_weights=Balanced, depth=4, l2_leaf_reg=3, learning_rate=0.05) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (613 s)


## 7. Synthèse : le meilleur réglage de chaque modèle

Pour chaque modèle, le meilleur réglage admis (surapprentissage de 7 % relatif au plus) sur l'ensemble des tours, avec son score en validation croisée, l'écart-type de ses 5 plis, son score en train et son surapprentissage. Alertes : ⚠️ variance si l'écart-type des plis dépasse 2 points ; ⚠️ aucun réglage admis.

In [7]:
lignes, meilleurs = [], {}
for nom, r in resultats.items():
    h = r["historique"]
    admis = h[h["admis"]]
    if admis.empty:
        ligne = h.loc[h["surapprentissage (% relatif)"].idxmin()]
        alerte = "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   f"Validation ({SCORE}, %)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2),
                   "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(h["tour"].max()), "Arrêt": r["raison"], "Durée (s)": r["duree"]})
display(pd.DataFrame(lignes).set_index("Modèle"))

,Réglage,"Validation (familles équilibrées, %)",Train (%),Surapprentissage (% relatif),Alerte,Tours,Arrêt,Durée (s)
Modèle,,,,,,,,
Régression logistique,"C=10, class_weight=balanced, l1_ratio=1",75.66 ± 0.48,75.84,0.25,—,1,tour 1 : meilleur réglage stable (à l'intérieu...,75
Arbre de décision,"class_weight=balanced, criterion=gini, max_dep...",76.30 ± 0.71,78.14,2.36,—,3,tour 3 : meilleur réglage stable (à l'intérieu...,7
RandomForest,"class_weight=balanced_subsample, max_depth=10,...",77.75 ± 0.71,81.26,4.32,—,6,tour 6 : meilleur réglage stable (à l'intérieu...,2270
MLP,"alpha=0.1, hidden_layer_sizes=(128, 64)",76.89 ± 0.29,77.93,1.33,—,4,tour 4 : meilleur réglage stable (à l'intérieu...,57
CatBoost,"auto_class_weights=Balanced, depth=4, l2_leaf_...",77.71 ± 0.57,80.85,3.88,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,613


**Les variables annulées par la régression logistique retenue** : elle est réapprise sur tout le train avec son meilleur réglage ; pour chaque variable d'origine, le nombre de ses coefficients (un par classe, et un par valeur pour les codifications) et la part mise à zéro. Avec une pénalité L2 (`l1_ratio` = 0), aucun coefficient n'est annulé.

In [8]:
lr = clone(ESPACES["Régression logistique"]["modele"]).set_params(**meilleurs["Régression logistique"]).fit(X_train, y_train)
noms = lr[1].get_feature_names_out()   # colonnes après le prétraitement (codifications en une colonne par valeur)
variables = [n.split("__", 1)[1].rsplit("_", 1)[0] if n.startswith("categories__") else n.split("__", 1)[1] for n in noms]
coefficients = pd.DataFrame(lr[-1].coef_, columns=variables)
annulation = pd.DataFrame({"Coefficients": coefficients.shape[0] * pd.Series(variables).value_counts(),
                           "Mis à zéro": (coefficients == 0).sum().groupby(level=0).sum()}).reindex(VARIABLES)
annulation["Part mise à zéro (%)"] = (annulation["Mis à zéro"] / annulation["Coefficients"] * 100).round(1)
print(f"Réglage : {libelle(meilleurs['Régression logistique'])}")
display(annulation)

Réglage : C=10, class_weight=balanced, l1_ratio=1


,Coefficients,Mis à zéro,Part mise à zéro (%)
PAY_2,20,1,5.0
PAY_3,20,3,15.0
PAY_4,20,6,30.0
PAY_5,20,6,30.0
PAY_6,20,6,30.0
LIMIT_BAL,5,1,20.0
BILL_AMT1,5,0,0.0
BILL_AMT2,5,0,0.0
BILL_AMT3,5,1,20.0
BILL_AMT4,5,1,20.0


## 8. Les modèles réglés, sur tout le train (prédictions hors pli)

Chaque modèle, avec son meilleur réglage, prédit chaque client du train une fois, par le pli qui ne l'a pas appris (`cross_val_predict`, mêmes 5 plis). Les tableaux portent sur tous les clients du train de cet essai, codifiés 1 compris.

In [9]:
predictions_hors_pli = {}
for nom, params in meilleurs.items():
    modele = ESPACES[nom]["modele"].set_params(**params)
    predictions_hors_pli[nom] = np.asarray(cross_val_predict(modele, X_train, y_train, cv=cv, n_jobs=ESPACES[nom]["n_jobs"], params=ESPACES[nom].get("fit_params", {}))).ravel().astype(int)
    print(f"{nom} : fait")
predictions_hors_pli["Règle « même codification qu'en août »"] = X_train["PAY_2"].to_numpy()
predictions_hors_pli["Toujours 0"] = np.zeros(len(X_train), dtype=int)

lignes = []
for nom, p in predictions_hors_pli.items():
    b = bonnes_predictions(y_train, p)
    lignes.append({"Modèle": nom, "Bonnes prédictions (%)": round(b["global"], 1),
                   **{f"Famille {NOMS_FAMILLES[f]} (%)": round(b[f], 1) for f in FAMILLES}})
display(pd.DataFrame(lignes).set_index("Modèle"))

Régression logistique : fait
Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


,Bonnes prédictions (%),Famille -2 (%),Famille -1 (%),Famille 0 (%),Famille 1 (%),Famille 2 et plus (%)
Modèle,,,,,,
Régression logistique,83.0,86.2,81.6,94.5,59.1,56.9
Arbre de décision,83.4,78.0,86.2,92.7,72.0,52.7
RandomForest,85.3,80.2,87.7,95.5,71.9,53.5
MLP,85.5,76.9,86.7,97.2,73.4,50.3
CatBoost,84.6,83.2,88.4,93.8,68.2,55.0
Règle « même codification qu'en août »,78.7,92.9,81.6,96.9,0.0,67.7
Toujours 0,49.2,0.0,0.0,100.0,0.0,0.0


**Comment lire** : une ligne par modèle réglé, sur les prédictions hors pli du train ; les deux dernières lignes sont les points de comparaison (la règle naïve ne peut jamais prédire 1 : il n'y a plus de codification 1 en août). La colonne « Famille 1 » est la part des vrais 1 que le modèle retrouve.

**Les changements entre août et septembre** : le même tableau que dans les notebooks 01 à 01f, avec deux lignes de plus pour les deux sortes de codifications 1 de septembre : **1 posé après un retard** (2 et plus en août) et **1 posé après une codification saine** (-2, -1 ou 0 en août). Un modèle peut retrouver une sorte de 1 et pas l'autre : ce ne serait pas un échec, mais une indication sur ce que chaque sorte de 1 signifie.

In [10]:
famille_aout, famille_sept = famille_pay(X_train["PAY_2"]), famille_pay(y_train)
sain_aout, sain_sept = famille_aout != 2, np.isin(famille_sept, [-2, -1, 0])
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & sain_sept,
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": sain_aout & (famille_sept == 2),
    "Changement entre codifications saines (par exemple 0 → -1)": sain_aout & sain_sept & (famille_aout != famille_sept),
    "1 posé après un retard (2 et plus en août, 1 en septembre)": (famille_aout == 2) & (famille_sept == 1),
    "1 posé après une codification saine (sain en août, 1 en septembre)": sain_aout & (famille_sept == 1),
    "Tous les clients qui changent": famille_aout != famille_sept,
}
lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : retrouvés (%)"] = round((famille_pay(p)[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

,Clients,Régression logistique : retrouvés (%),Arbre de décision : retrouvés (%),RandomForest : retrouvés (%),MLP : retrouvés (%),CatBoost : retrouvés (%),Règle « même codification qu'en août » : retrouvés (%)
Type de changement,,,,,,,
"Sortie du retard (2 et plus en août, sain en septembre)",366,23.2,43.2,58.7,64.8,61.2,0.0
Retard maintenu (2 et plus en août et en septembre ; pas un changement),1695,79.2,70.9,76.6,74.2,76.3,100.0
"Nouveau retard (sain en août, 2 et plus en septembre)",809,10.0,14.6,4.9,0.1,10.1,0.0
Changement entre codifications saines (par exemple 0 → -1),998,5.6,10.0,8.0,14.1,12.4,0.0
"1 posé après un retard (2 et plus en août, 1 en septembre)",1467,50.9,62.0,61.1,57.8,60.2,0.0
"1 posé après une codification saine (sain en août, 1 en septembre)",1461,67.4,82.0,82.6,89.0,76.2,0.0
Tous les clients qui changent,5101,38.3,48.7,47.8,49.6,47.6,0.0


**Comment lire** : une ligne par type de changement entre août et septembre, avec le nombre de clients du train et, pour chaque modèle, la part de ces clients rangés dans leur vraie famille de septembre. La règle naïve est à 0 % sur les changements (elle ne prédit jamais de 1) et à 100 % sur les retards maintenus. Les deux lignes « 1 posé » se lisent séparément : chacune dit si cette sorte de 1 suit une logique que les modèles apprennent.

## 9. La codification 1 : retrouvée, ou pas ?

Pour chaque modèle réglé, sur les prédictions hors pli du train :
- **retrouvés** : part des vrais 1 que le modèle codifie 1 ;
- **précision** : parmi les clients que le modèle codifie 1, part de vrais 1 ;
- **ce que deviennent les vrais 1 ratés** : la famille que le modèle leur donne à la place ;
- **selon la codification d'août** : la même mesure pour les 1 posés après un retard (`PAY_2` >= 2) et après une codification saine (`PAY_2` <= 0), les deux familles vues dans l'étude de la codification 1.

In [11]:
vrai = famille_pay(y_train)
famille_aout = famille_pay(X_train["PAY_2"])
vrais_1 = vrai == 1
lignes, devenir, par_aout = [], [], []
for nom, p in predictions_hors_pli.items():
    if nom in ("Toujours 0", "Règle « même codification qu'en août »"):
        continue
    f = famille_pay(p)
    predits_1 = f == 1
    lignes.append({"Modèle": nom, "Vrais 1": int(vrais_1.sum()), "Codifiés 1 par le modèle": int(predits_1.sum()),
                   "Dont vrais 1": int((predits_1 & vrais_1).sum()),
                   "Retrouvés (%)": round((f[vrais_1] == 1).mean() * 100, 1),
                   "Précision (%)": round((vrai[predits_1] == 1).mean() * 100, 1) if predits_1.any() else np.nan})
    rates = f[vrais_1 & ~predits_1]
    devenir.append({"Modèle": nom, "Vrais 1 ratés": len(rates),
                    **{f"Prédits {NOMS_FAMILLES[c]} (%)": round((rates == c).mean() * 100, 1) for c in FAMILLES if c != 1}})
    for libelle_aout, masque in [("après un retard (PAY_2 >= 2)", famille_aout == 2), ("après une codification saine (PAY_2 <= 0)", famille_aout != 2)]:
        m1 = vrais_1 & masque
        p1 = predits_1 & masque
        par_aout.append({"Modèle": nom, "1 posés": libelle_aout, "Vrais 1": int(m1.sum()),
                         "Retrouvés (%)": round((f[m1] == 1).mean() * 100, 1),
                         "Précision (%)": round((vrai[p1] == 1).mean() * 100, 1) if p1.any() else np.nan})
display(Markdown("**Codification 1 retrouvée**"))
display(pd.DataFrame(lignes).set_index("Modèle"))
display(Markdown("**Ce que deviennent les vrais 1 ratés**"))
display(pd.DataFrame(devenir).set_index("Modèle"))
display(Markdown("**Selon la codification d'août**"))
display(pd.DataFrame(par_aout).set_index(["Modèle", "1 posés"]))

**Codification 1 retrouvée**

,Vrais 1,Codifiés 1 par le modèle,Dont vrais 1,Retrouvés (%),Précision (%)
Modèle,,,,,
Régression logistique,2928,2547,1731,59.1,68.0
Arbre de décision,2928,3095,2107,72.0,68.1
RandomForest,2928,2925,2104,71.9,71.9
MLP,2928,3038,2149,73.4,70.7
CatBoost,2928,2740,1997,68.2,72.9


**Ce que deviennent les vrais 1 ratés**

,Vrais 1 ratés,Prédits -2 (%),Prédits -1 (%),Prédits 0 (%),Prédits 2 et plus (%)
Modèle,,,,,
Régression logistique,1197,33.5,7.7,0.3,58.6
Arbre de décision,821,24.8,9.5,0.5,65.2
RandomForest,824,23.1,11.9,0.4,64.7
MLP,779,12.3,14.9,1.4,71.4
CatBoost,931,30.5,10.4,0.3,58.8


**Selon la codification d'août**

Vrais 1  \
Modèle                1 posés                                              
Régression logistique après un retard (PAY_2 >= 2)                  1467   
                      après une codification saine (PAY_2 <= 0)     1461   
Arbre de décision     après un retard (PAY_2 >= 2)                  1467   
                      après une codification saine (PAY_2 <= 0)     1461   
RandomForest          après un retard (PAY_2 >= 2)                  1467   
                      après une codification saine (PAY_2 <= 0)     1461   
MLP                   après un retard (PAY_2 >= 2)                  1467   
                      après une codification saine (PAY_2 <= 0)     1461   
CatBoost              après un retard (PAY_2 >= 2)                  1467   
                      après une codification saine (PAY_2 <= 0)     1461   

                                                                 Retrouvés (%)  \
Modèle                1 posés                                                    
Régression logistique après un retard (PAY_2 >= 2)                        50.9   
                      après une codification saine (PAY_2 <= 0)           67.4   
Arbre de décision     après un retard (PAY_2 >= 2)                        62.0   
                      après une codification saine (PAY_2 <= 0)           82.0   
RandomForest          après un retard (PAY_2 >= 2)                        61.1   
                      après une codification saine (PAY_2 <= 0)           82.6   
MLP                   après un retard (PAY_2 >= 2)                        57.8   
                      après une codification saine (PAY_2 <= 0)           89.0   
CatBoost              après un retard (PAY_2 >= 2)                        60.2   
                      après une codification saine (PAY_2 <= 0)           76.2   

                                                                 Précision (%)  
Modèle                1 posés                                                   
Régression logistique après un retard (PAY_2 >= 2)                        57.4  
                      après une codification saine (PAY_2 <= 0)           79.0  
Arbre de décision     après un retard (PAY_2 >= 2)                        60.5  
                      après une codification saine (PAY_2 <= 0)           75.3  
RandomForest          après un retard (PAY_2 >= 2)                        64.9  
                      après une codification saine (PAY_2 <= 0)           78.3  
MLP                   après un retard (PAY_2 >= 2)                        63.9  
                      après une codification saine (PAY_2 <= 0)           76.1  
CatBoost              après un retard (PAY_2 >= 2)                        65.2  
                      après une codification saine (PAY_2 <= 0)           80.4

**Comment lire** : le premier tableau dit si le 1 se prédit (retrouvés) et si le modèle le pose à bon escient (précision) ; le deuxième, vers quelle codification le modèle envoie les 1 qu'il ne reconnaît pas ; le troisième sépare les 1 posés après un retard de ceux posés après une codification saine. Un 1 bien retrouvé, avec une bonne précision, suit une logique que les montants et les mois précédents portent ; un 1 rarement retrouvé, ou posé à tort sur beaucoup d'autres clients, dépend d'autre chose.

## 10. L'importance des variables

**Importance par permutation** : pour chaque variable, ses valeurs sont mélangées entre les clients de la validation, et l'on mesure **la baisse du score** (moyenne des familles, en points). Une variable dont le mélange ne fait rien perdre n'est pas utilisée par le modèle. Le calcul se fait pli par pli, sur les clients que le modèle n'a pas appris (mêmes 5 plis, 5 mélanges par variable), puis la moyenne des plis est affichée. Les variables d'origine sont mélangées entières (une codification avec toutes ses valeurs).

Le second tableau regroupe les variables **par mois** : septembre (M : `BILL_AMT1`, `PAY_AMT1`), août (M-1 : `PAY_2`, `BILL_AMT2`, `PAY_AMT2`), juillet (M-2), juin (M-3), mai (M-4), avril (M-5), et le plafond. Il répond à la question : les mois anciens servent-ils à quelque chose ?

In [12]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance

REGLAGES = meilleurs
NOMS_MOIS = {1: "septembre (M)", 2: "août (M-1)", 3: "juillet (M-2)", 4: "juin (M-3)", 5: "mai (M-4)", 6: "avril (M-5)"}

def mois_de(variable):
    return "plafond" if variable == "LIMIT_BAL" else NOMS_MOIS[int(variable[-1])]

importances = {}
for nom, params in REGLAGES.items():
    par_pli = []
    for apprentissage, validation in cv.split(X_train, y_train):
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_train.iloc[apprentissage], y_train.iloc[apprentissage])
        r = permutation_importance(m, X_train.iloc[validation], y_train.iloc[validation], scoring=scoreur,
                                   n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=X_train.columns)
    print(f"{nom} : fait")

table = pd.DataFrame(importances)
table["Moyenne des modèles"] = table.mean(axis=1)
display(Markdown("**Par variable** (baisse du score en points, de la plus importante à la moins importante en moyenne)"))
display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
par_mois = table.groupby(table.index.map(mois_de)).sum()
ordre = [NOMS_MOIS[n] for n in range(1, 7)] + ["plafond"]
display(Markdown("**Par mois** (somme des variables du mois)"))
display(par_mois.reindex(ordre).round(2))

Régression logistique : fait


Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


**Par variable** (baisse du score en points, de la plus importante à la moins importante en moyenne)

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
PAY_2,47.09,48.25,51.29,37.29,36.44,44.07
BILL_AMT1,12.37,13.32,10.72,14.23,11.28,12.39
PAY_3,3.46,5.64,3.78,1.91,1.54,3.27
PAY_AMT1,2.05,1.89,1.98,2.20,2.74,2.17
BILL_AMT2,4.03,0.46,1.34,1.04,1.40,1.65
PAY_AMT2,2.55,0.02,0.77,1.96,0.94,1.25
PAY_6,1.21,0.79,0.34,0.79,0.61,0.75
PAY_5,0.57,1.32,0.34,0.64,0.58,0.69
PAY_4,0.96,1.02,0.13,0.90,0.39,0.68
BILL_AMT5,0.62,1.09,0.72,0.51,0.20,0.63


**Par mois** (somme des variables du mois)

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
septembre (M),14.42,15.22,12.70,16.43,14.02,14.56
août (M-1),53.67,48.73,53.40,40.29,38.78,46.97
juillet (M-2),4.51,5.66,5.14,3.42,2.71,4.29
juin (M-3),1.48,1.51,1.20,1.96,1.24,1.48
mai (M-4),1.66,2.43,1.46,1.68,1.16,1.68
avril (M-5),1.54,1.82,0.81,1.49,1.13,1.36
plafond,0.27,0.22,0.25,0.46,0.76,0.39


**Comment lire** : une ligne par variable (ou par mois), une colonne par modèle ; la valeur est la baisse du score, en points, quand la variable est mélangée. Proche de 0 : le modèle ne s'en sert pas ; négative : le mélange ne fait rien perdre (bruit). Si les mois de juin, mai et avril sont proches de 0 pour tous les modèles, les colonnes anciennes n'apportent rien à la codification de septembre.